# Hands-on Exercises

These exercises reinforce the tasks that actually appear in the source Fresh Eyes notebooks. The final optional task is explicitly marked as an extension.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

import numpy as np
import xarray as xr
from workshop import open_tas

DATA = 'data/demo/tas_demo.nc'
tas = open_tas(DATA)

## Exercise 1: variable and time selection with NCO

Create a file containing only `tas` for 1981–2010. The demo dataset begins in January 1850, so the inclusive time indices are 1,572 through 1,931. Verify that the result contains 360 monthly time steps.

In [ ]:
%%bash
set -euo pipefail
mkdir -p outputs/exercises
# Complete and run the command:
ncks -O -v tas -d time,1572,1931 data/demo/tas_demo.nc outputs/exercises/tas_1981_2010.nc
test "$(cdo -s ntime outputs/exercises/tas_1981_2010.nc)" -eq 360
echo 'PASS: 360 monthly steps selected.'

## Exercise 2: seasonal climatology with CDO

Calculate a four-step seasonal climatology in degrees Celsius. Then inspect it with xarray.

In [ ]:
%%bash
set -euo pipefail
cdo -L -yseasmean -subc,273.15 data/demo/tas_demo.nc outputs/exercises/seasonal_degC.nc
test "$(cdo -s ntime outputs/exercises/seasonal_degC.nc)" -eq 4
echo 'PASS: four seasonal climatology fields created.'

In [ ]:
seasonal = xr.open_dataset('outputs/exercises/seasonal_degC.nc')['tas']
seasonal.plot(col='time', col_wrap=2, cmap='magma', robust=True)

## Exercise 3: change the anomaly baseline

Calculate global annual anomalies relative to 1961–1990 with both CDO and xarray. Your comparison should use an explicit tolerance. Start from the commands in `02_climate_stripes.ipynb`.

In [ ]:
# Python reference
weights = np.cos(np.deg2rad(tas.lat))
annual = tas.weighted(weights).mean(('lat', 'lon')).groupby('time.year').mean('time')
python_6190 = annual - annual.sel(year=slice(1961, 1990)).mean('year')
python_6190.plot()

## Optional extension: a regional mean

Spatial subsetting is not present in the source notebooks. This optional extension shows how the same architecture can continue beyond the original material.

Use `cdo sellonlatbox,100,160,-10,10` before `fldmean` to calculate a tropical western Pacific mean, then compare it with xarray selection and cosine-latitude weighting.

## Reflection

For each exercise, identify:

1. which implementation is shorter;
2. which implementation makes the algorithm easiest to customize;
3. which metadata checks are necessary before trusting the result;
4. whether the task belongs in preprocessing, interpretation, or visualization.